# 📊 Global Primates Watch — Análise Exploratória de Dados (EDA)

Este notebook realiza uma análise exploratória completa:
- Estatísticas descritivas por categoria de risco
- Análise por continente e bioma
- Correlação entre área de distribuição e risco
- Identificação de hotspots de biodiversidade
- Análise de sobreposição de espécies ameaçadas

## 📦 Importações e Carregamento de Dados

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_utils import get_statistics_by_category, get_statistics_by_continent

# Configurar estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Importações concluídas!")

✓ Importações concluídas!


In [2]:
# Carregar dados processados
geojson_path = project_root / 'data' / 'processed' / 'primates_map.geojson'
csv_path = project_root / 'data' / 'processed' / 'primates_species_clean.csv'

print(f"Carregando dados de: {geojson_path}")
gdf = gpd.read_file(str(geojson_path))

print(f"Carregando dados de: {csv_path}")
df = pd.read_csv(str(csv_path))

print(f"\n✓ Dados carregados com sucesso!")
print(f"  - Registros: {len(df)}")
print(f"  - Colunas: {len(df.columns)}")

Carregando dados de: c:\Users\eferro\Desktop\Global Primates Watch\data\processed\primates_map.geojson
Carregando dados de: c:\Users\eferro\Desktop\Global Primates Watch\data\processed\primates_species_clean.csv

✓ Dados carregados com sucesso!
  - Registros: 526
  - Colunas: 31


## 📈 Estatísticas Descritivas Gerais

In [3]:
print("=" * 70)
print("ESTATÍSTICAS GERAIS DE PRIMATAS AMEAÇADOS")
print("=" * 70)

print(f"\n📊 Resumo Geral:")
print(f"  - Total de espécies: {len(df)}")
print(f"  - Espécies ameaçadas (EN/CR): {len(df[df['category'].isin(['EN', 'CR'])])}")
print(f"  - Espécies vulneráveis (VU): {len(df[df['category'] == 'VU'])}")
print(f"  - Espécies pouco preocupantes (LC): {len(df[df['category'] == 'LC'])}")

print(f"\n⚠️ Distribuição por Categoria IUCN:")
category_dist = df['category'].value_counts().sort_index()
for cat, count in category_dist.items():
    pct = count / len(df) * 100
    print(f"  - {cat}: {count:3d} espécies ({pct:5.1f}%)")

print(f"\n⚠️ Distribuição por Nível de Risco:")
risk_dist = df['risco'].value_counts()
for risk, count in risk_dist.items():
    pct = count / len(df) * 100
    print(f"  - {risk}: {count:3d} espécies ({pct:5.1f}%)")

ESTATÍSTICAS GERAIS DE PRIMATAS AMEAÇADOS

📊 Resumo Geral:
  - Total de espécies: 526
  - Espécies ameaçadas (EN/CR): 234
  - Espécies vulneráveis (VU): 115
  - Espécies pouco preocupantes (LC): 118

⚠️ Distribuição por Categoria IUCN:
  - CR:  88 espécies ( 16.7%)
  - DD:  13 espécies (  2.5%)
  - EN: 146 espécies ( 27.8%)
  - EX:   2 espécies (  0.4%)
  - LC: 118 espécies ( 22.4%)
  - NT:  44 espécies (  8.4%)
  - VU: 115 espécies ( 21.9%)

⚠️ Distribuição por Nível de Risco:
  - Alto Risco: 234 espécies ( 44.5%)
  - Baixo Risco: 162 espécies ( 30.8%)
  - Médio Risco: 115 espécies ( 21.9%)
  - Desconhecido:  13 espécies (  2.5%)
  - Crítico:   2 espécies (  0.4%)


## 🌍 Análise por Continente

In [4]:
# Calcular estatísticas por continente
continent_stats = get_statistics_by_continent(gdf)

print("\n🌍 Distribuição de Espécies por Continente:")
continent_df = pd.DataFrame(list(continent_stats.items()), columns=['Continente', 'Espécies'])
continent_df = continent_df.sort_values('Espécies', ascending=False)
print(continent_df.to_string(index=False))

print(f"\n📍 Análise Detalhada:")
for continent, count in sorted(continent_stats.items(), key=lambda x: x[1], reverse=True):
    pct = count / len(gdf) * 100
    print(f"  - {continent}: {count} espécies ({pct:.1f}%)")


🌍 Distribuição de Espécies por Continente:
Continente  Espécies
    África       167
      Ásia       157
   América       116
   Oceania        86

📍 Análise Detalhada:
  - África: 167 espécies (31.7%)
  - Ásia: 157 espécies (29.8%)
  - América: 116 espécies (22.1%)
  - Oceania: 86 espécies (16.3%)


## 🔴 Espécies em Risco Crítico

In [5]:
# Análise de espécies criticamente ameaçadas
critical_species = df[df['category'].isin(['CR', 'EX', 'EW'])].copy()

print(f"\n🔴 Espécies em Risco Crítico (CR/EX/EW):")
print(f"  - Total: {len(critical_species)} espécies")

if len(critical_species) > 0:
    print(f"\nPrimeiras 15 espécies em risco crítico:")
    critical_display = critical_species[['sci_name', 'category', 'category_pt']].head(15)
    print(critical_display.to_string(index=False))
    
    print(f"\nDistribuição por categoria:")
    print(critical_species['category'].value_counts())


🔴 Espécies em Risco Crítico (CR/EX/EW):
  - Total: 90 espécies

Primeiras 15 espécies em risco crítico:
               sci_name category            category_pt
BRACHYTELES ARACHNOIDES       CR Criticamente em Perigo
BRACHYTELES HYPOXANTHUS       CR Criticamente em Perigo
 PLECTUROCEBUS OENANTHE       CR Criticamente em Perigo
  PLECTUROCEBUS OLALLAE       CR Criticamente em Perigo
   CALLITHRIX FLAVICEPS       CR Criticamente em Perigo
  SAPAJUS XANTHOSTERNOS       CR Criticamente em Perigo
    CEBUS AEQUATORIALIS       CR Criticamente em Perigo
       CEBUS TRINITATIS       CR Criticamente em Perigo
   CERCOCEBUS GALERITUS       CR Criticamente em Perigo
  CERCOPITHECUS ROLOWAY       CR Criticamente em Perigo
     COLOBUS VELLEROSUS       CR Criticamente em Perigo
         EULEMUR MONGOZ       CR Criticamente em Perigo
    EULEMUR CINEREICEPS       CR Criticamente em Perigo
     EULEMUR FLAVIFRONS       CR Criticamente em Perigo
        GORILLA GORILLA       CR Criticamente em Perigo

## 📊 Análise de Área de Distribuição vs. Risco

In [6]:
# Calcular área de cada polígono em km²
gdf_area = gdf.copy()
# O GeoJSON já possui as colunas category e risco!
gdf_area['area_km2'] = gdf_area.geometry.area / 1e6  # Converter para km²

print("\n📏 Análise de Área de Distribuição:")
print(f"\nEstatísticas de área por nível de risco:")
area_stats = gdf_area.groupby('risco')['area_km2'].agg(['count', 'mean', 'median', 'min', 'max'])
area_stats.columns = ['Espécies', 'Área Média (km²)', 'Mediana (km²)', 'Mín (km²)', 'Máx (km²)']
print(area_stats.to_string())

print(f"\n💡 Observação:")
print(f"Espécies com menor área de distribuição tendem a ser mais ameaçadas.")
print(f"Isso indica que a fragmentação de habitat é um fator crítico de risco.")



📏 Análise de Área de Distribuição:

Estatísticas de área por nível de risco:
              Espécies  Área Média (km²)  Mediana (km²)     Mín (km²)  Máx (km²)
risco                                                                           
Alto Risco         234          0.000006   7.273722e-07  4.694099e-10   0.000215
Baixo Risco        162          0.000049   1.339032e-05  2.464699e-08   0.000674
Crítico              2          0.000026   2.593875e-05  9.392641e-07   0.000051
Desconhecido        13          0.000010   2.385415e-06  1.441665e-08   0.000100
Médio Risco        115          0.000013   3.180459e-06  4.694099e-10   0.000115

💡 Observação:
Espécies com menor área de distribuição tendem a ser mais ameaçadas.
Isso indica que a fragmentação de habitat é um fator crítico de risco.


C:\Users\eferro\AppData\Local\Temp\ipykernel_17708\1631999110.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_area['area_km2'] = gdf_area.geometry.area / 1e6  # Converter para km²


## 🔥 Hotspots de Biodiversidade

In [7]:
# Identificar regiões com alta concentração de espécies ameaçadas
threatened = gdf_area[gdf_area['risco'].isin(['Alto Risco', 'Crítico'])].copy()

# Criar grid de análise (aproximado por latitude/longitude)
threatened['lat_bin'] = pd.cut(threatened.geometry.centroid.y, bins=10)
threatened['lon_bin'] = pd.cut(threatened.geometry.centroid.x, bins=10)

# Contar espécies por célula
hotspots = threatened.groupby(['lat_bin', 'lon_bin']).size().reset_index(name='count')
hotspots = hotspots.sort_values('count', ascending=False)

print("\n🔥 Hotspots de Espécies Ameaçadas (Top 10):")
print(hotspots.head(10).to_string(index=False))

print(f"\n💡 Interpretação:")
print(f"Essas regiões concentram múltiplas espécies ameaçadas e devem ser prioridades")
print(f"para conservação e proteção de habitat.")


🔥 Hotspots de Espécies Ameaçadas (Top 10):
          lat_bin            lon_bin  count
(-19.47, -13.624]   (39.406, 60.939]     36
(-25.375, -19.47]   (39.406, 60.939]     29
(-13.624, -7.777]   (39.406, 60.939]     13
  (-1.931, 3.915] (104.005, 125.538]     13
   (3.915, 9.762]   (-3.659, 17.874]     11
  (-1.931, 3.915]  (82.472, 104.005]     10
 (15.608, 21.454] (104.005, 125.538]      8
  (9.762, 15.608] (104.005, 125.538]      6
 (15.608, 21.454]  (82.472, 104.005]      6
   (3.915, 9.762] (-90.006, -68.258]      6

💡 Interpretação:
Essas regiões concentram múltiplas espécies ameaçadas e devem ser prioridades
para conservação e proteção de habitat.


C:\Users\eferro\AppData\Local\Temp\ipykernel_17708\1474408921.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  threatened['lat_bin'] = pd.cut(threatened.geometry.centroid.y, bins=10)
C:\Users\eferro\AppData\Local\Temp\ipykernel_17708\1474408921.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  threatened['lon_bin'] = pd.cut(threatened.geometry.centroid.x, bins=10)


## 📌 Sobreposição de Espécies Ameaçadas

In [8]:
# Análise de sobreposição espacial
print("\n📌 Análise de Sobreposição:")

# Contar quantas espécies ameaçadas compartilham a mesma região
threatened_regions = threatened.copy()
threatened_regions['region_id'] = threatened_regions['lat_bin'].astype(str) + '_' + threatened_regions['lon_bin'].astype(str)

overlap_analysis = threatened_regions.groupby('region_id').agg({
    'sci_name': 'count',
    'category': lambda x: list(x)
}).rename(columns={'sci_name': 'num_especies'})

overlap_analysis = overlap_analysis.sort_values('num_especies', ascending=False)

print(f"\nRegiões com múltiplas espécies ameaçadas (Top 5):")
for idx, (region_id, row) in enumerate(overlap_analysis.head(5).iterrows(), 1):
    print(f"  {idx}. Região {region_id}: {row['num_especies']} espécies")

print(f"\n📊 Distribuição de sobreposição:")
print(f"  - Regiões com 1 espécie: {(overlap_analysis['num_especies'] == 1).sum()}")
print(f"  - Regiões com 2-5 espécies: {((overlap_analysis['num_especies'] >= 2) & (overlap_analysis['num_especies'] <= 5)).sum()}")
print(f"  - Regiões com 6+ espécies: {(overlap_analysis['num_especies'] >= 6).sum()}")


📌 Análise de Sobreposição:

Regiões com múltiplas espécies ameaçadas (Top 5):
  1. Região (-19.47, -13.624]_(39.406, 60.939]: 36 espécies
  2. Região (-25.375, -19.47]_(39.406, 60.939]: 29 espécies
  3. Região (-1.931, 3.915]_(104.005, 125.538]: 13 espécies
  4. Região (-13.624, -7.777]_(39.406, 60.939]: 13 espécies
  5. Região (3.915, 9.762]_(-3.659, 17.874]: 11 espécies

📊 Distribuição de sobreposição:
  - Regiões com 1 espécie: 8
  - Regiões com 2-5 espécies: 26
  - Regiões com 6+ espécies: 10


## 🎯 Conclusões da Análise Exploratória

In [9]:
print("\n" + "="*70)
print("CONCLUSÕES PRINCIPAIS")
print("="*70)

print(f"\n1️⃣ SITUAÇÃO GERAL:")
threatened_pct = len(df[df['risco'].isin(['Alto Risco', 'Crítico'])]) / len(df) * 100
print(f"   - {threatened_pct:.1f}% das espécies de primatas estão ameaçadas")
print(f"   - Situação crítica para conservação global")

print(f"\n2️⃣ FATORES DE RISCO:")
print(f"   - Espécies com menor área de distribuição são mais ameaçadas")
print(f"   - Fragmentação de habitat é um fator crítico")
print(f"   - Sobreposição de espécies em regiões específicas")

print(f"\n3️⃣ PRIORIDADES DE CONSERVAÇÃO:")
print(f"   - Proteger hotspots de biodiversidade identificados")
print(f"   - Focar em espécies criticamente ameaçadas (CR/EX)")
print(f"   - Expandir áreas de distribuição protegidas")

print(f"\n4️⃣ PRÓXIMOS PASSOS:")
print(f"   - Integrar dados de desmatamento")
print(f"   - Análise de mudanças climáticas")
print(f"   - Modelagem preditiva de risco")

print("\n" + "="*70)


CONCLUSÕES PRINCIPAIS

1️⃣ SITUAÇÃO GERAL:
   - 44.9% das espécies de primatas estão ameaçadas
   - Situação crítica para conservação global

2️⃣ FATORES DE RISCO:
   - Espécies com menor área de distribuição são mais ameaçadas
   - Fragmentação de habitat é um fator crítico
   - Sobreposição de espécies em regiões específicas

3️⃣ PRIORIDADES DE CONSERVAÇÃO:
   - Proteger hotspots de biodiversidade identificados
   - Focar em espécies criticamente ameaçadas (CR/EX)
   - Expandir áreas de distribuição protegidas

4️⃣ PRÓXIMOS PASSOS:
   - Integrar dados de desmatamento
   - Análise de mudanças climáticas
   - Modelagem preditiva de risco

